# Phase 3: Exploratory Data Analysis & Data Understanding
## Project: Road Traffic Accident Severity Prediction & Risk Factor Identification

This notebook conducts comprehensive exploratory data analysis on the **US Accidents** dataset to uncover:
1. Missing value patterns and schema distributions
2. Target distribution and severity class imbalance
3. Temporal trends (Hour of day, rush hours, weekend patterns)
4. Environmental & weather risk conditions (Visibility, precipitation, temperature)
5. Road infrastructure correlations (Junctions, traffic signals, crossings)


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_DATA_DIR, TARGET_COLUMN
from src.data_loader import load_dataset

# Plot formatting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11


### 1. Load Dataset

In [ ]:
df = load_dataset(sample_size=30000)
print("Dataset Shape:", df.shape)
df.head()


### 2. Schema and Missing Values Inspection

In [ ]:
missing_stats = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage (%)': (df.isnull().sum() / len(df)) * 100
}).sort_values(by='Missing Percentage (%)', ascending=False)

missing_stats.head(15)


### 3. Severity Distribution & Class Imbalance Analysis
Notice the extreme rarity of severe/fatal accidents.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Original 4-class severity
df['Severity'].value_counts(normalize=True).sort_index().plot(
    kind='bar', color='#3b82f6', edgecolor='black', ax=ax1
)
ax1.set_title("Original Severity Distribution (Classes 1-4)")
ax1.set_xlabel("Severity Level")
ax1.set_ylabel("Proportion")

# Binary target (Serious vs Non-Serious)
is_serious = (df['Severity'] >= 3).astype(int)
is_serious.value_counts(normalize=True).rename({0: 'Non-Serious (1-2)', 1: 'Serious (3-4)'}).plot(
    kind='bar', color=['#10b981', '#ef4444'], edgecolor='black', ax=ax2
)
ax2.set_title("Binary Target Distribution")
ax2.set_ylabel("Proportion")

plt.tight_layout()
plt.show()
print(f"Serious Class Proportion: {is_serious.mean()*100:.2f}%")


### 4. Severity vs Temporal Drivers (Hour, Rush Hour, Day of Week)

In [ ]:
df['Start_Time'] = pd.to_datetime(df['Start_Time'])
df['Hour'] = df['Start_Time'].dt.hour
df['DayOfWeek'] = df['Start_Time'].dt.day_name()
df['Is_Serious'] = (df['Severity'] >= 3).astype(int)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Hourly risk
hourly_risk = df.groupby('Hour')['Is_Serious'].mean() * 100
hourly_risk.plot(kind='line', marker='o', color='#dc2626', lw=2.5, ax=ax1)
ax1.set_title("Serious Accident Rate by Hour of Day (%)")
ax1.set_xlabel("Hour (0-23)")
ax1.set_ylabel("Serious Rate (%)")
ax1.grid(True)

# Day of week risk
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_risk = df.groupby('DayOfWeek')['Is_Serious'].mean().reindex(order) * 100
day_risk.plot(kind='bar', color='#4f46e5', edgecolor='black', ax=ax2)
ax2.set_title("Serious Accident Rate by Day of Week (%)")
ax2.set_ylabel("Serious Rate (%)")

plt.tight_layout()
plt.show()


### 5. Severity vs Weather & Visibility

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x='Severity', y='Visibility(mi)', data=df, palette='Blues')
plt.title("Visibility (miles) Distribution across Severity Levels")
plt.ylim(0, 11)
plt.show()
